# DEM Error Estimation with Spurt

This notebook demonstrates how spurt estimates and applies DEM error corrections during 3D phase unwrapping. We use pre-run results from a **Capella X-band** dataset over **Mexico City** (11 dates, 30 days, perpendicular baselines from -789 to +635 m) to visualize the estimation outputs.

The notebook does **not** re-run the full EMCF pipeline. It shows how to:

1. Build and inspect the design matrix from baseline metadata
2. Visualize the interferogram network in time-baseline space
3. Interpret the estimated DEM error and velocity maps
4. Compare unwrapped phase with and without model guidance

For the theory behind these steps, see the [DEM Error and Velocity Estimation](../theory/dem-error-velocity.md) page.

**Reproducing these results:**
```bash
# The results were produced by running dolphin + spurt with DEM error estimation enabled.
# See the dolphin_config.yaml in the data directory for the exact configuration.
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from spurt.links import build_design_matrix

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# Path to pre-run results (symlinked into docs/notebooks/data/)
DATA_DIR = Path("data/mexico_city")
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}. Create a symlink."

## Load baselines and build the design matrix

The design matrix $\mathbf{A}$ relates model parameters (velocity, DEM error) to interferometric phase. Each row corresponds to one interferogram; column 0 encodes velocity sensitivity and column 1 encodes DEM error sensitivity.

In [ ]:
# Load per-SLC baselines
slc_df = pd.read_csv(DATA_DIR / "baselines_per_slc.csv", parse_dates=["date"])
dates = slc_df["date"].values.astype("datetime64[D]")
bperp_m = slc_df["bperp_m"].values

print(f"Number of SLCs: {len(dates)}")
print(f"Date range: {dates[0]} to {dates[-1]}")
print(f"Bperp range: {bperp_m.min():.0f} to {bperp_m.max():.0f} m")
slc_df

In [ ]:
# Load interferogram baselines to get the network edges
ifg_df = pd.read_csv(DATA_DIR / "baselines.csv")

# Extract SLC index pairs from the filenames
# Map date strings to SLC indices
date_to_idx = {str(d): i for i, d in enumerate(dates)}

ref_dates = pd.to_datetime(ifg_df["reference_time_utc"]).dt.strftime("%Y-%m-%d")
sec_dates = pd.to_datetime(ifg_df["secondary_time_utc"]).dt.strftime("%Y-%m-%d")
ifg_edges = np.array(
    [[date_to_idx[r], date_to_idx[s]] for r, s in zip(ref_dates, sec_dates)]
)
print(f"Number of interferograms: {len(ifg_edges)}")
bperp_min = ifg_df["bperp_m"].min()
bperp_max = ifg_df["bperp_m"].max()
print(f"Bperp range in ifgs: {bperp_min:.0f} to {bperp_max:.0f} m")

In [ ]:
# Capella X-band parameters
WAVELENGTH_M = 0.031  # 3.1 cm X-band
SLANT_RANGE_M = 550_000
INCIDENCE_RAD = 0.61  # ~35 degrees

amat = build_design_matrix(
    ifg_edges=ifg_edges,
    dates=dates,
    bperp_m=bperp_m,
    wavelength_m=WAVELENGTH_M,
    slant_range_m=SLANT_RANGE_M,
    incidence_rad=INCIDENCE_RAD,
)

print(f"Design matrix shape: {amat.shape}")
vel_min, vel_max = amat[:, 0].min(), amat[:, 0].max()
dem_min, dem_max = amat[:, 1].min(), amat[:, 1].max()
print(f"Velocity sensitivity range: {vel_min:.4f} to {vel_max:.4f} rad/(mm/yr)")
print(f"DEM error sensitivity range: {dem_min:.4f} to {dem_max:.4f} rad/m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Temporal baselines in days
delta_t = np.array(
    [(dates[s] - dates[r]).astype(float) for r, s in ifg_edges]
)
delta_bperp = np.array(
    [bperp_m[s] - bperp_m[r] for r, s in ifg_edges]
)

ax = axes[0]
ax.scatter(delta_t, amat[:, 0], c="steelblue", s=30, edgecolor="k", linewidth=0.5)
ax.set_xlabel("Temporal baseline (days)")
ax.set_ylabel("Sensitivity (rad per mm/yr)")
ax.set_title("Velocity sensitivity")
ax.axhline(0, color="gray", linewidth=0.5)

ax = axes[1]
ax.scatter(delta_bperp, amat[:, 1], c="firebrick", s=30, edgecolor="k", linewidth=0.5)
ax.set_xlabel("Perpendicular baseline difference (m)")
ax.set_ylabel("Sensitivity (rad per m)")
ax.set_title("DEM error sensitivity")
ax.axhline(0, color="gray", linewidth=0.5)

fig.tight_layout()
plt.show()

## Interferogram network

The interferogram network is typically a Hop-3 graph in time-Bperp space: each SLC is connected to its 3 nearest temporal neighbors. The wide spread of perpendicular baselines in this dataset gives strong DEM error sensitivity.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Plot edges
for r, s in ifg_edges:
    ax.plot(
        [dates[r], dates[s]],
        [bperp_m[r], bperp_m[s]],
        "k-", linewidth=0.8, alpha=0.5,
    )

# Plot SLC dates as points
ax.scatter(dates, bperp_m, c="steelblue", s=60, zorder=5, edgecolor="k", linewidth=0.5)
for _i, (d, b) in enumerate(zip(dates, bperp_m)):
    ax.annotate(
        f"{b:.0f} m", (d, b),
        textcoords="offset points", xytext=(5, 5),
        fontsize=7, color="gray",
    )

ax.set_xlabel("Date")
ax.set_ylabel("Perpendicular baseline (m)")
ax.set_title("Interferogram network in time-Bperp space")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Load pre-run results

The pixelwise DEM error and velocity maps were computed by spurt's `write_link_params()` function, which integrates per-edge gradients to per-pixel values via weighted least-squares.

In [ ]:
# Load pre-computed results
results = np.load(DATA_DIR / "pixelwise_dem_error.npz")
dem_error = results["dem_error_m"]
velocity = results["velocity_mm_yr"]

print(f"DEM error shape: {dem_error.shape}")
print(f"Velocity shape: {velocity.shape}")

# Load temporal coherence
tcoh_path = DATA_DIR / "interferograms" / "temporal_coherence_20240626_20240726.tif"
with rasterio.open(tcoh_path) as src:
    tcoh = src.read(1)

print(f"Temporal coherence shape: {tcoh.shape}")

## DEM error map

The estimated DEM error shows spatial structure correlated with topography. Positive values indicate the DEM underestimates true elevation; negative values indicate overestimation.

In [ ]:
# Mask NaN values for statistics
valid = np.isfinite(dem_error)
vmax = np.nanpercentile(np.abs(dem_error[valid]), 95)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(dem_error, cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
cbar = fig.colorbar(im, ax=ax, shrink=0.8, label="DEM error (m)")
ax.set_title("Estimated DEM error")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
fig.tight_layout()
plt.show()

## DEM error distribution

The DEM error distribution is expected to be roughly centered near zero with standard deviation depending on the DEM quality and scene topography.

In [ ]:
from scipy.stats import norm

valid_dem = dem_error[valid]
mu, sigma = np.nanmean(valid_dem), np.nanstd(valid_dem)
p5, p50, p95 = np.nanpercentile(valid_dem, [5, 50, 95])

fig, ax = plt.subplots(figsize=(7, 4))
counts, bins, _ = ax.hist(
    valid_dem, bins=200, density=True, alpha=0.7, color="steelblue", edgecolor="none",
    range=(-vmax, vmax),
)

# Gaussian fit overlay
x_fit = np.linspace(-vmax, vmax, 500)
ax.plot(x_fit, norm.pdf(x_fit, mu, sigma), "r-", linewidth=1.5, label="Gaussian fit")

ax.set_xlabel("DEM error (m)")
ax.set_ylabel("Density")
ax.set_title("DEM error distribution")
stats_text = (
    f"Mean: {mu:.2f} m\n"
    f"Std:  {sigma:.2f} m\n"
    f"Median: {p50:.2f} m\n"
    f"5th/95th: {p5:.1f} / {p95:.1f} m"
)
ax.text(
    0.97, 0.95, stats_text, transform=ax.transAxes,
    fontsize=9, verticalalignment="top", horizontalalignment="right",
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "wheat", "alpha": 0.8},
)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## DEM error vs temporal coherence

High-coherence pixels have well-constrained DEM error estimates (small spread), while low-coherence pixels show a wide range of estimated values. This 2D histogram illustrates the relationship.

In [ ]:
# Use the same spatial extent for both arrays
valid_both = valid & np.isfinite(tcoh) & (tcoh > 0)

fig, ax = plt.subplots(figsize=(7, 5))
h = ax.hist2d(
    tcoh[valid_both].ravel(),
    dem_error[valid_both].ravel(),
    bins=[100, 200],
    range=[[0, 1], [-vmax, vmax]],
    cmap="inferno",
    cmin=1,
)
fig.colorbar(h[3], ax=ax, label="Count")
ax.set_xlabel("Temporal coherence")
ax.set_ylabel("DEM error (m)")
ax.set_title("DEM error vs temporal coherence")
fig.tight_layout()
plt.show()

## Unwrapped phase: with vs without model

For a large-Bperp interferogram (2024-06-26 to 2024-07-02, $B_\perp = -789$ m), the model captures the long-wavelength DEM-correlated signal. Spurt writes three files per interferogram:

- `*.unw.tif` -- full unwrapped phase
- `*.unw_model.tif` -- model component (velocity + DEM error prediction)
- `*.unw_diff.tif` -- max difference between tiles (overlap consistency diagnostic)

In [ ]:
# Load the large-Bperp pair
pair = "20240626_20240702"
output_dir = DATA_DIR / "spurt_output"

with rasterio.open(output_dir / f"{pair}.unw.tif") as src:
    unw = src.read(1)
with rasterio.open(output_dir / f"{pair}.unw_model.tif") as src:
    unw_model = src.read(1)

# Residual = total unwrapped - model
residual = unw - unw_model

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, data, title, cmap in zip(
    axes,
    [unw, unw_model, residual],
    ["Unwrapped phase", "Model component", "Residual (unw - model)"],
    ["RdBu_r", "RdBu_r", "RdBu_r"],
):
    vabs = np.nanpercentile(np.abs(data[np.isfinite(data)]), 95)
    im = ax.imshow(data, cmap=cmap, vmin=-vabs, vmax=vabs, interpolation="nearest")
    fig.colorbar(im, ax=ax, shrink=0.7, label="rad")
    ax.set_title(title)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

fig.suptitle(f"Interferogram {pair} (Bperp = -789 m)", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## Summary

Key takeaways from this tutorial:

- The design matrix translates physical baselines (time, perpendicular baseline) into phase sensitivities for velocity and DEM error.
- DEM error estimation is performed per-edge on spatial gradients, naturally canceling common atmospheric signals.
- The estimated model is used to guide phase wrapping disambiguation, reducing the number of residues MCF must resolve.
- Temporal coherence serves as both the optimization objective and a quality metric for the final estimates.

For the mathematical derivation behind each step, see the [DEM Error and Velocity Estimation](../theory/dem-error-velocity.md) theory page.

For API details, see the [spurt.links](../reference/spurt/links/) reference documentation.